# 06. GNSS Blackout Simulation & Classical Inertial Dead Reckoning Baseline

**Objective**: Simulate GNSS denial windows, double-integrate raw IMU measurements, and quantify exponential drift.

## 1. Run Baseline across Multiple Outage Durations (10s, 30s, 60s, 120s)

In [ ]:
import sys
sys.path.insert(0, '../..')
import yaml
import pandas as pd
from Data_details.src.dataset_loader import DatasetLoader
from Data_details.src.inertial_baseline import run_inertial_baseline

with open('../config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
loader = DatasetLoader('../../IO-VNBD-master', '../../Data_details/data/raw')
s_df, _ = loader.load_sequence(cfg['selected_sequence']['smartphone_file'])

metrics_list = []
for dur in [10.0, 30.0, 60.0, 120.0]:
    res = run_inertial_baseline(s_df, blackout_start_s=150.0, blackout_duration_s=dur)
    metrics_list.append(res['metrics'])

metrics_df = pd.DataFrame(metrics_list)
metrics_df[['blackout_duration_s', 'distance_travelled_m', 'endpoint_error_m', 'rmse_m', 'drift_pct']]

## 2. Visual Comparison: Raw INS vs Reference Path

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(mpimg.imread('../../Data_details/outputs/plots/raw_ins_vs_reference.png'))
axes[0].axis('off')
axes[0].set_title('Estimated vs Reference Path (60s Blackout)')
axes[1].imshow(mpimg.imread('../../Data_details/outputs/plots/position_error_vs_time.png'))
axes[1].axis('off')
axes[1].set_title('Position Error Growth Over Time')
plt.tight_layout()
plt.show()

## 3. Findings
- Raw inertial dead reckoning exhibits severe exponential drift: 49.2% drift at 10s (61.8m error), 60.2% drift at 60s (284.9m error), and 405.9m error at 120s.
- SIH 2026 goal requires <10% drift (<5m over 50m, <100m over 1km).
- Root causes: Accelerometer bias, gravity leakage due to uncalibrated phone attitude, vibration noise, and lack of kinematic/speed constraints.
- Phase 2 must introduce attitude calibration, vibration filtering, and Phase 4 AI/ML velocity estimation.